In [ ]:
import copy

import torch
import torch.nn as nn
import torchvision
import numpy as np
from matplotlib import pyplot as plt

from confidence.unsupervised.classic.prototype import ClassPrototypeConfidence
from search.parallel_gradient import ParallelGradientDescent
from utils.eval.vis import plt_setup_latex
from utils.sampling import BatchNegativeSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
#look for experiment files in parents
import os
path_found = False
current_path = os.getcwd()
while not path_found:
    if os.path.exists(os.path.join(current_path, "experiment_files")):
        path_found = True
        break
    current_path = os.path.dirname(current_path)


In [ ]:
experiment_files_path_data = os.path.join(current_path, "experiment_files", "data")

In [ ]:
dataset ="si_score"
architecture = "resnet50_pretrained"

In [ ]:
from experiment_thesis.dataset_preperation.get_dataset import get_dataset_info,get_dataset

dataset_info = get_dataset_info(dataset)
dataset_dict = get_dataset(dataset_info,path=experiment_files_path_data)


In [ ]:
dataset_train = dataset_dict['train_dataset']
dataset_val = dataset_dict['val_dataset']
dataset_test = dataset_dict['test_dataset']
train_loader = dataset_dict['train_loader']
val_loader = dataset_dict['val_loader']
test_loader = dataset_dict['test_loader']
n_classes = dataset_info.num_classes
train_loader_transformed = dataset_dict['train_loader_transformed']
val_loader_transformed = dataset_dict['val_loader_transformed']
test_loader_transformed = dataset_dict['test_loader_transformed']
train_loader_no_shuffle = dataset_dict['train_loader_no_shuffle']



In [ ]:
print(len(dataset_train))

In [ ]:
print(len(train_loader))


In [ ]:
dataset_train[2][0].shape

In [ ]:
batch_size = next(iter(train_loader))[0].shape[0]

In [ ]:
#test images
fig, axs = plt.subplots(nrows=3, ncols=1, figsize=(20, 10))

axs[0].imshow(torchvision.utils.make_grid(next(iter(train_loader))[0], nrow=batch_size//4).permute(1, 2, 0).cpu())
axs[1].imshow(torchvision.utils.make_grid(next(iter(val_loader))[0], nrow=batch_size//4).permute(1, 2, 0).cpu())
axs[2].imshow(torchvision.utils.make_grid(next(iter(test_loader_transformed))[0], nrow=batch_size//4).permute(1, 2, 0).cpu())
axs[0].set_title('Training')
axs[1].set_title('Validation')
axs[2].set_title('Test')



In [ ]:
from experiment_thesis.main import train_and_get_model
from experiment_thesis.dataset_preperation.basic_networks import get_network
from utils.eval.main_model import evaluate_base_model

model_dir_path = os.path.join(current_path, "experiment_files", "models")
embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")

model = get_network(dataset_info,architecture, num_classes=n_classes).to(device)
modelname = f"{dataset}_{architecture}"
cache_name_train= f"{dataset}_{architecture}_embedding_cache_train"

#if si score then no train
if not "si_score" in dataset:
    train_and_get_model(model,model_dir_path,modelname, train_loader, val_loader , trainer_kwargs= {
            "accelerator": "auto",
            "max_epochs": 100,
            "precision": "16-mixed",
    },load_if_exists=True)


#res = evaluate_base_model(model, test_loader_transformed, device)
#print(res)
model.eval().cuda()

In [ ]:
#check main model
res = evaluate_base_model(model, test_loader_transformed, device)
print(res)
res = evaluate_base_model(model, test_loader, device)
print(res)

In [ ]:
import confidence.direct.logit_based
import utils.affine_transforms

energy = confidence.direct.logit_based.EnergyConfidence()
from utils.transform_sequence import TransformSequence

transformation_sequence = TransformSequence(
    transformations=[
        utils.affine_transforms.AffineTransformation2D.ROTATION.value,
    ],
    domains=[(-torch.pi, torch.pi)],
    device=device,init_method="sobol"
)


In [ ]:
test_loader_transformed = torch.utils.data.DataLoader(test_loader_transformed.dataset, batch_size=8, shuffle=False, num_workers=4,pin_memory=True,persistent_workers=True)

In [ ]:
transform_seq = transformation_sequence.cuda()

In [ ]:
model

In [ ]:
from torchinfo import summary
summary(model, input_size=(1, 3, 240, 240), device=device.type)

In [ ]:

from utils.transformation_problem import TransformationProblem
from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence,PredictedSplitConfidence
from confidence.unsupervised.classic.nn_pytorch import KNNConfidence, PerClassKNNConfidence

from confidence.input_transform import InputTransformImage, PCAInputModule, RandomProjectionModule

In [ ]:
from search.shgo import SHGO

di = SHGO(
    initial_samples=17,
    local_runs=1,
    local_max_steps=0,
)

In [ ]:
transform_name ="rotation"

In [ ]:
model_dir_path = os.path.join(current_path, "experiment_files", "models")
embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")
# Add results dir and helper for save paths
results_dir_path = os.path.join(current_path, "experiment_files", "results", dataset, architecture, "correct_or_all_k_1_v2")
os.makedirs(results_dir_path, exist_ok=True)


def savepath(label: str) -> str:
    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(results_dir_path,transform_name, f"{safe}.json")

In [ ]:
firt_image = next(iter(train_loader_no_shuffle))[0].to(device)
plt.imshow(firt_image[1].permute(1, 2, 0).cpu())

In [ ]:
from experiment_thesis.dataset_preperation.get_dataset import get_layer_embedding_cache_config,create_layer_embedding_cache
cache_config = get_layer_embedding_cache_config(dataset, architecture,transform_name=None,dataset_info=dataset_info)
train_cache =create_layer_embedding_cache(model, train_loader_no_shuffle,cache_config, embedding_cache_path, device=device)

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from experiment_thesis.dataset_preperation.basic_networks import get_network_layer
layer,layer_io = get_network_layer(dataset_info,architecture,0)

In [ ]:
from torch.utils.data import SequentialSampler
from embedding_cache import LayerEmbeddingCache
cache_train = train_cache


cache_name_train = f"{dataset}_{architecture}_{transform_name}_embedding_cache_train"


embeddings_t, final_t, classes_t = cache_train.get_correct_embeddings(layer, capture_modes=layer_io, flatten=True)
dual_output_model = cache_train.make_wrapper(layer, capture_modes=layer_io, concat=False, flatten=True)

embeddings_t_all, final_t_all, classes_t_all = cache_train.__call__(layer, capture_modes=layer_io, flatten=True)




from utils.transformation_problem import TransformationProblem
from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence, PredictedSplitConfidence
from confidence.unsupervised.classic.nn_pytorch import KNNConfidence, PerClassKNNConfidence

from confidence.input_transform import InputTransformImage, PCAInputModule, RandomProjectionModule

nn_pytorch_pretrained = KNNConfidence(metric="cosine", input_transform=None,k=1)
nn_pytorch_pretrained.fit(embeddings_t, classes_t)
nn_pytorch_pretrained.to(device)

conf_split_pretrained = PredictedSplitConfidence(nn_pytorch_pretrained, EnergyConfidence(), mult=False, b=0.0)
conf_mod_nn_pytorch_pretrained = SinglePassConfidence(dual_output_model, conf_split_pretrained, index=1)
problem_nn_pytorch_pretrained = TransformationProblem(conf_mod_nn_pytorch_pretrained, transform_seq,
                                                      consolidate_method="consolidate_simple")
model.eval().to(device)


nn_pytorch_pretrained_all = KNNConfidence(metric="cosine", input_transform=None,k=1)
nn_pytorch_pretrained_all.fit(embeddings_t_all, classes_t_all)
nn_pytorch_pretrained_all.to(device)
conf_split_pretrained_all = PredictedSplitConfidence(nn_pytorch_pretrained_all, EnergyConfidence(), mult=False, b=0.0)
conf_mod_nn_pytorch_pretrained_all = SinglePassConfidence(dual_output_model, conf_split_pretrained_all, index=1)
problem_nn_pytorch_pretrained_all = TransformationProblem(conf_mod_nn_pytorch_pretrained_all, transform_seq,
                                                      consolidate_method="consolidate_simple")



In [ ]:
embeddings_t.shape


In [ ]:
embeddings_t_all.shape

In [ ]:
from search.shgo import SHGO

random_search = SHGO(initial_samples=60, local_max_steps=0)
#if si score only use 17 samples
if dataset == "si_score":
    random_search = SHGO(initial_samples=17, local_max_steps=0)


In [ ]:
energy_confidence = SinglePassConfidence(model, EnergyConfidence())

In [ ]:
from utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search

#res_energy = load_or_run_evaluate_confidence_and_search(
#    model, optimizer=random_search, problem=TransformationProblem(energy_confidence, transform_seq,
 #                                                                  consolidate_method="consolidate_simple"),
#    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
 #   save_path=savepath("energy"), show_progress=True,
 #   repeats=1)

In [ ]:
repeats=10 if dataset != "si_score" else 4

In [ ]:
shuffled_test_loader_transformed = torch.utils.data.DataLoader(
    test_loader_transformed.dataset,
    batch_size=test_loader_transformed.batch_size,
    shuffle=True,
    num_workers=test_loader_transformed.num_workers,
    pin_memory=True,
    persistent_workers=True if test_loader_transformed.num_workers > 0 else False,
)

In [ ]:

res_correct = load_or_run_evaluate_confidence_and_search(
    model, optimizer=random_search, problem=problem_nn_pytorch_pretrained,
    test_loader=shuffled_test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("knn"), show_progress=True,
    repeats=repeats)

res_all = load_or_run_evaluate_confidence_and_search(
    model, optimizer=random_search, problem=problem_nn_pytorch_pretrained_all,
    test_loader=shuffled_test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("knn_all"), show_progress=True,
    repeats=repeats)

In [ ]:
print("Using only correct embeddings")
print(res_correct)


In [ ]:
print("Using all embeddings")
print(res_all)

In [ ]:
W = plt_setup_latex()

In [ ]:
#plot mean and se against each other
mean_correct = res_correct["accuracy_mean"]
se_correct = res_correct["accuracy_se"]
mean_all = res_all["accuracy_mean"]
se_all = res_all["accuracy_se"]
#plot using bar with error bars
labels = ["Correct Embeddings", "All Embeddings"]
means = [mean_correct, mean_all]
ses = [se_correct, se_all]
x = np.arange(len(labels))
width = 0.35
fig, ax = plt.subplots()
bars = ax.bar(x, means, width, yerr=ses, capsize=5, color=['blue', 'orange'])
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy by Embedding Type')
ax.set_xticks(x)
ax.set_xticklabels(labels)
#set limit based on means and ses so that we are -0.05 to +0.05 around the highest bar
max_height = max(means[i] + ses[i] for i in range(len(means)))
min_height = min(means[i] - ses[i] for i in range(len(means)))
ax.set_ylim([min_height - 0.05, max_height + 0.05])
#add value on top of bars


